In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 8 &mdash; Signals as Vectors, Systems as Matrices: the SVD</h1></center>

In Module 7 every LTI system was described by one function, its impulse response, and deconvolution failed spectacularly wherever that response's
spectrum was nearly zero. Once a signal is sampled it is simply a list of $N$ numbers, a vector, and every linear system acting on it, convolution
included, becomes a matrix. This module uses that viewpoint to ask <i>when</i> a system can be undone. Its main tool is the <b>singular value
decomposition (SVD)</b>: every matrix rotates, stretches and rotates again, and the stretches (the singular values) tell us which parts of a signal
survive the system and which are lost in the noise. We use it to stabilize Module 7's deconvolution, to compress an image, and to separate coherent
arrivals from noise in a seismic shot gather. No linear algebra is assumed beyond multiplying a matrix by a vector.

# Outline
- [8.1 Motivation: Why Matrices?](#8.1-Motivation:-Why-Matrices?)
- [8.2 Signals as Vectors, Systems as Matrices](#8.2-Signals-as-Vectors,-Systems-as-Matrices)
  - [8.2.1 A sampled signal is a vector](#8.2.1-A-sampled-signal-is-a-vector)
  - [8.2.2 Linearity: every linear system is a matrix](#8.2.2-Linearity:-every-linear-system-is-a-matrix)
  - [8.2.3 Time invariance: the convolution matrix](#8.2.3-Time-invariance:-the-convolution-matrix)
- [8.3 The Adjoint](#8.3-The-Adjoint)
- [8.4 The Geometry of a Matrix: the SVD](#8.4-The-Geometry-of-a-Matrix:-the-SVD)
  - [8.4.1 Rotate, stretch, rotate](#8.4.1-Rotate,-stretch,-rotate)
  - [8.4.2 Singular values are gains](#8.4.2-Singular-values-are-gains)
  - [8.4.3 The SVD of a convolution matrix](#8.4.3-The-SVD-of-a-convolution-matrix)
- [8.5 Rank, Conditioning and the Pseudo-inverse](#8.5-Rank,-Conditioning-and-the-Pseudo-inverse)
- [8.6 Low-Rank Approximation](#8.6-Low-Rank-Approximation)
- [8.7 Geophysical Application: SVD Filtering of a Shot Gather](#8.7-Geophysical-Application:-SVD-Filtering-of-a-Shot-Gather)
  - [8.7.1 A synthetic gather](#8.7.1-A-synthetic-gather)
  - [8.7.2 A real gather: CAPshot](#8.7.2-A-real-gather:-CAPshot)
- [8.8 Parting Thoughts](#8.8-Parting-Thoughts)
- [Microquestions](#Microquestions)

In [ ]:
# Helper functions and the Module 7 deconvolution experiment used throughout this module
import matplotlib.image as mpimg
from matplotlib.collections import LineCollection

def convmat(h, N):
    """(N+L-1) x N convolution matrix: column j is the impulse response h shifted down j samples."""
    h = np.asarray(h, float); L = len(h)
    H = np.zeros((N + L - 1, N))
    for j in range(N):
        H[j:j+L, j] = h
    return H

def ricker(t, f0):
    """Ricker (Mexican-hat) wavelet with peak frequency f0 [Hz]."""
    a = (np.pi*f0*t)**2
    return (1 - 2*a)*np.exp(-a)

def gray(path, block):
    """Read an image, average the color channels, and block-average it by `block` pixels."""
    img = mpimg.imread(path).astype(float)
    img = img.mean(axis=2) if img.ndim == 3 else img
    m, n = (img.shape[0]//block)*block, (img.shape[1]//block)*block
    img = img[:m, :n].reshape(m//block, block, n//block, block).mean(axis=(1, 3))
    return img/img.max()

# Module 7's experiment: a boxcar blurred by a Gaussian, plus noise of rms 1e-3
nbox = 300
x_box = np.repeat([0., 0., 1., 0., 0.], nbox//5)
tg = np.arange(50.)
g = np.exp(-((tg - 25)/4)**2); g /= g.sum()          # 50-tap Gaussian blur, unit sum
Hg = convmat(g, nbox)                                 # 349 x 300 convolution matrix
y_clean = Hg @ x_box
y_obs = y_clean + 1e-3*np.random.default_rng(404).standard_normal(len(y_clean))
Ug, sg, Vgt = np.linalg.svd(Hg, full_matrices=False)  # the SVD used in Sections 8.4 and 8.5

<a name="8.1"></a>
## 8.1 Motivation: Why Matrices?

Figure 8.1 repeats Module 7's deconvolution experiment (Section 7.9): a boxcar $x$ is blurred by a 50-sample Gaussian $g$, and a little noise $n$
of rms $10^{-3}$ is added, so the data are $y = g * x + n$. Dividing the data spectrum by $G(f)$ should undo the blur.

In [ ]:
C = DSP_COLORS
Nf = len(y_obs)
G = np.fft.fft(g, Nf); f = np.fft.rfftfreq(Nf)
noise = y_obs - y_clean
x_naive = np.real(np.fft.ifft(np.fft.fft(y_obs) / G))[:nbox]
err = np.linalg.norm(x_naive - x_box) / np.linalg.norm(x_box)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
axs[0].plot(x_box, color=C['input'], label='truth $x$')
axs[0].plot(y_obs, color=C['output'], label='data $y = g*x + n$')
axs[0].set_xlabel('sample $n$'); axs[0].set_title('(a) Boxcar blurred by a Gaussian'); axs[0].legend(loc='upper right', fontsize=10)
axs[0].set_ylim(-0.15, 1.3)
axs[1].semilogy(f, np.abs(G[:len(f)]), color=C['impulse_resp'], label='$|G(f)|$')
axs[1].semilogy(f, np.abs(np.fft.rfft(y_clean)), color=C['output'], label='$|Y(f)|$ without noise')
axs[1].semilogy(f, np.abs(np.fft.rfft(noise)), color=C['noise'], lw=1, label='$|N(f)|$, noise')
axs[1].set_ylim(1e-20, 1e4); axs[1].set_yticks(10.0**np.arange(-18, 4, 3)); axs[1].set_xlim(0, 0.5)
axs[1].set_xlabel('frequency [cycles/sample]'); axs[1].set_title('(b) Spectra'); axs[1].legend(loc='lower left', fontsize=9)
axs[2].plot(x_naive, color=C['output'], lw=1)
axs[2].set_yscale('symlog', linthresh=10); axs[2].set_yticks([-1e12, -1e8, -1e4, 0, 1e4, 1e8, 1e12])
axs[2].set_xlabel('sample $n$'); axs[2].set_title(f'(c) $Y/G$: relative error {err:.0e}')
plt.tight_layout(); plt.show()

<b>Figure 8.1.</b> Why naive deconvolution fails. (a) The boxcar (blue) and the blurred, noisy data (red). (b) The Gaussian's spectrum $|G(f)|$ (green) falls to 3e-17; beyond about 0.15 cycles/sample the noise spectrum (olive) is larger than the signal (red). (c) Dividing by $G$ amplifies that noise by up to $10^{16}$: the "deconvolved" boxcar has a relative error of 5e+12 (note the symmetric-log axis). <i>(Python-generated.)</i>

Write the blur as a matrix acting on the vector of samples, $\mathbf{y} = \mathbf{Hx} + \mathbf{n}$ (Section 8.2). The question "can the blur be
undone?" becomes "can $\mathbf{H}$ be inverted, and what can we do when it effectively cannot?" The rest of the module builds the tools to answer it.

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication.</b> This is the experiment of Module 7's deconvolution section with its constant floor (<code>DC_term</code>)
set to zero. Spectral division divides each Fourier coefficient of the data by $G(\omega)$, and $G$ is the Fourier transform of a Gaussian, so it decays like
a Gaussian (Module 5) and reaches $10^{-17}$ at high frequency. Module 7 hid the problem by adding a constant to $g$; this module explains why that worked,
and does it properly.
</div>

<a name="8.2"></a>
## 8.2 Signals as Vectors, Systems as Matrices

<a name="8.2.1"></a>
### 8.2.1 A sampled signal is a vector

A signal of $N$ samples, $x[0], x[1], \dots, x[N-1]$, is a list of $N$ numbers, so we can write it as a column vector $\mathbf{x}$ with $N$ components.
Two vectors are combined with the dot product $\mathbf{x}\cdot\mathbf{y} = \mathbf{x}^T\mathbf{y} = \sum_n x[n]\,y[n]$, where the superscript $T$
(transpose) turns a column into a row. The length of a signal vector is $\|\mathbf{x}\| = \sqrt{\mathbf{x}^T\mathbf{x}}$.

<div class="alert alert-info">
<b>Note:</b> The dot product $\mathbf{x}\cdot\mathbf{y} = \sum_n x[n]y[n]$ is the PHGN 100 product $\mathbf{F}\cdot\mathbf{d}$ that gives work, applied to
vectors with $N$ components instead of 3. Its square, $\|\mathbf{x}\|^2 = \sum_n x[n]^2$, is the energy that Module 5's Parseval relation and Module 7's
autocorrelation at zero lag both compute.
</div>

<a name="8.2.2"></a>
### 8.2.2 Linearity: every linear system is a matrix

A sampled signal is a sum of shifted unit impulses, $\mathbf{x} = \sum_j x[j]\,\boldsymbol{\delta}_j$, where $\boldsymbol{\delta}_j$ is 1 at sample $j$ and
0 elsewhere. If the system is linear, its output is the same sum of the responses to each impulse. Collect those responses as the columns of a matrix
$\mathbf{H}$: column $j$ is the output for input $\boldsymbol{\delta}_j$. Then

$$\mathbf{y} = \mathbf{H}\mathbf{x}, \qquad y[m] = \sum_j H_{mj}\,x[j] \tag{1}$$

Every linear system acting on sampled signals is a matrix, and the matrix–vector product is superposition.

<a name="8.2.3"></a>
### 8.2.3 Time invariance: the convolution matrix

If the system is also time invariant, the response to $\boldsymbol{\delta}_j$ is the impulse response $h$ shifted by $j$ samples. Every column is then the
same $h$ shifted one row further down, so the entries depend only on the difference $m - j$:

$$H_{mj} = h[m - j], \qquad y[m] = \sum_j h[m - j]\,x[j] = (h * x)[m] \tag{2}$$

This is the convolution sum of Module 7. A matrix that is constant along its diagonals is called a <b>Toeplitz</b> matrix. For an impulse response of
$L = 3$ samples acting on $N = 4$ input samples, the full output has $N + L - 1 = 6$ samples:

$$\begin{pmatrix} y[0]\\ y[1]\\ y[2]\\ y[3]\\ y[4]\\ y[5]\end{pmatrix} =
\begin{pmatrix} h[0] & 0 & 0 & 0\\ h[1] & h[0] & 0 & 0\\ h[2] & h[1] & h[0] & 0\\ 0 & h[2] & h[1] & h[0]\\ 0 & 0 & h[2] & h[1]\\ 0 & 0 & 0 & h[2]\end{pmatrix}
\begin{pmatrix} x[0]\\ x[1]\\ x[2]\\ x[3]\end{pmatrix} \tag{3}$$

Figure 8.2 builds a $33 \times 29$ convolution matrix one column at a time, for a 5-sample pulse $h$ and an input $\mathbf{x}$ with four
spikes. Watch two things. On the left, each new column (inside the orange frame) is the same pulse shifted one sample further down, which is why the matrix is
constant along its diagonals. On the right, the output grows as $x[j]$ times that column is added: when $x[j] = 0$ nothing changes, and each spike adds a
scaled copy of $h$. This is Module 7's spike-by-spike picture of convolution, read off the matrix.

In [ ]:
C = DSP_COLORS
h = np.array([-0.25, 0.5, 1, 0.5, -0.25])              # 5-tap zero-mean pulse
xs = np.zeros(29); xs[[4, 11, 14, 22]] = [1, -0.6, 0.8, 0.5]
H = convmat(h, 29)                                       # 33 x 29
y = H @ xs
n_frames = 30                                            # 29 build frames (one column each) + 1 hold frame
fig, (axa, axb) = plt.subplots(1, 2, figsize=(8.8, 3.6), gridspec_kw=dict(width_ratios=[1, 1.4]))
im = axa.imshow(np.full_like(H, np.nan), cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
axa.set_xlabel('column $j$ (input sample)'); axa.set_ylabel('row $m$ (output sample)')
axa.set_xticks(range(0, 29, 5)); axa.set_yticks(range(0, 33, 5)); axa.grid(False)
box = plt.Rectangle((-0.5, -0.5), 1, 33, fill=False, ec=C['highlight'], lw=2.5)
axa.add_patch(box)
axb.plot(np.arange(33), y, color=C['output'], lw=1, alpha=0.35, label='final $\\mathbf{y} = \\mathbf{Hx}$')
stems = axb.vlines(np.arange(33), 0, np.zeros(33), color=C['output'], lw=2)
dots, = axb.plot(np.arange(33), np.zeros(33), 'o', color=C['output'], ms=5, label='running sum $\\sum_{i\\leq j} x[i]\\,\\mathbf{h}_i$')
axb.axhline(0, color='k', lw=0.6)
axb.set_xlim(-0.8, 32.8); axb.set_ylim(-1.0, 1.25); axb.set_xlabel('output sample $m$'); axb.legend(loc='upper right', fontsize=9)
title = fig.suptitle('', fontsize=13)

def update(k):
    j = min(k, 28)                                       # column j is new in this frame
    shown = np.full_like(H, np.nan); shown[:, :j + 1] = H[:, :j + 1]
    im.set_data(shown)
    box.set_x(j - 0.5); box.set_visible(k < 29)
    part = H[:, :j + 1] @ xs[:j + 1]
    stems.set_segments([[(m, 0), (m, v)] for m, v in enumerate(part)])
    dots.set_ydata(part)
    title.set_text(f'column {j} (h shifted down {j} rows) times x[{j}] = {xs[j]:+.1f}'
                   if k < 29 else 'H complete: y = Hx is the convolution x * h')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=400, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 8.2.</b> Building the convolution matrix. (a) The matrix $\mathbf{H}$ (red positive, blue negative); column $j$ is the pulse $h = [-0.25, 0.5, 1, 0.5, -0.25]$ shifted down $j$ rows. (b) The running sum $\sum_{i\le j} x[i]\,\mathbf{h}_i$ (stems), which reaches the final output $\mathbf{y} = \mathbf{Hx}$ (faint line) when all 29 columns have been added. 30 frames, one new column per frame. <i>(Python-generated.)</i>

<a name="8.3"></a>
## 8.3 The Adjoint

The transpose $\mathbf{H}^T$ swaps rows and columns. For a real matrix it is also the <b>adjoint</b>: the unique matrix that satisfies

$$\langle \mathbf{Hx}, \mathbf{y}\rangle = \langle \mathbf{x}, \mathbf{H}^T\mathbf{y}\rangle \quad \text{for all } \mathbf{x}, \mathbf{y} \tag{4}$$

where $\langle\mathbf{a}, \mathbf{b}\rangle = \mathbf{a}^T\mathbf{b}$. For the convolution matrix, row $n$ of $\mathbf{H}^T$ is column $n$ of $\mathbf{H}$, the
pulse starting at sample $n$, so

$$(\mathbf{H}^T\mathbf{y})[n] = \sum_k h[k]\,y[n + k] \tag{5}$$

which is the <b>crosscorrelation</b> of $y$ with $h$. Figure 8.3 checks this numerically, together with equation (4).

In [ ]:
C = DSP_COLORS
h = np.array([-0.25, 0.5, 1, 0.5, -0.25])
xs = np.zeros(20); xs[[3, 8, 10, 15]] = [1, -0.6, 0.8, 0.5]
H = convmat(h, 20); y = H @ xs
adj = H.T @ y
cor = np.correlate(y, h, 'valid')
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4))
axa.vlines(np.arange(24), 0, y, color=C['output'], lw=2); axa.plot(np.arange(24), y, 'o', color=C['output'], ms=4)
axa.axhline(0, color='k', lw=0.6); axa.set_title(r'(a) $\mathbf{y} = \mathbf{Hx}$ (24 samples)'); axa.set_xlabel('sample')
axb.vlines(np.arange(20), 0, adj, color=C['input'], lw=2)
axb.plot(np.arange(20), adj, 'o', color=C['input'], ms=5, label=r'$\mathbf{H}^T\mathbf{y}$ (matrix)')
axb.plot(np.arange(20), cor, '-', color=C['highlight'], lw=1.5, label=r"np.correlate(y, h, 'valid')")
axb.axhline(0, color='k', lw=0.6); axb.set_title(r'(b) $\mathbf{H}^T\mathbf{y}$ (20 samples)'); axb.set_xlabel('sample')
axb.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()
print(f'max |H^T y - correlate(y, h)| = {np.max(np.abs(adj - cor)):.1e}')
rng = np.random.default_rng(404); xr, yr = rng.standard_normal(20), rng.standard_normal(24)
print(f'dot-product test: <Hx, y> = {(H @ xr) @ yr:.12f},  <x, H^T y> = {xr @ (H.T @ yr):.12f}')

<b>Figure 8.3.</b> The adjoint of convolution is correlation. (a) The output $\mathbf{y} = \mathbf{Hx}$ of Figure 8.2. (b) $\mathbf{H}^T\mathbf{y}$ (blue stems) and NumPy's crosscorrelation of $y$ with $h$ (orange line) agree to machine precision. The printed dot-product test, equation (4) for a random pair of vectors, is the standard check that two codes really are adjoints of each other. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; systems as matrices.</b> Applying $\mathbf{H}^T$ crosscorrelates with $h$. That is Module 7's statement that correlation is convolution
with the time-reversed pulse, now read off the matrix: transposing swaps rows and columns, and that reverses the direction in which $h$ slides. It is also why
picking a traveltime by correlation (Module 7) is an <i>adjoint</i> operation, not an inverse: $\mathbf{H}^T\mathbf{H} \neq \mathbf{I}$ in general.
</div>

<a name="8.4"></a>
## 8.4 The Geometry of a Matrix: the SVD

<a name="8.4.1"></a>
### 8.4.1 Rotate, stretch, rotate

A $2\times 2$ matrix maps the unit circle to an ellipse. The <b>singular value decomposition</b> says that this is always done in three steps:

$$\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T \tag{6}$$

where $\mathbf{V}^T$ and $\mathbf{U}$ are rotations (possibly with a reflection), called <i>orthogonal</i> matrices because they keep lengths and right
angles, and $\boldsymbol{\Sigma} = \mathrm{diag}(\sigma_1, \sigma_2, \dots)$ stretches along the axes by the <b>singular values</b>
$\sigma_1 \ge \sigma_2 \ge \dots \ge 0$. The columns $\mathbf{v}_i$ of $\mathbf{V}$ are the input directions and the columns $\mathbf{u}_i$ of $\mathbf{U}$
are the output directions:

$$\mathbf{A}\mathbf{v}_i = \sigma_i\,\mathbf{u}_i, \qquad \mathbf{A}^T\mathbf{u}_i = \sigma_i\,\mathbf{v}_i \tag{7}$$

The same three steps hold for any matrix, of any shape: $\mathbf{U}$ and $\mathbf{V}$ are then rotations in more dimensions.

Figure 8.4 applies $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$ to the unit circle one factor at a time, with $\mathbf{V}$ a
rotation by $30^\circ$, $\boldsymbol{\Sigma} = \mathrm{diag}(2, 0.5)$ and $\mathbf{U}$ a rotation by $60^\circ$. Watch the two arrows, which start as the
input directions $\mathbf{v}_1$ and $\mathbf{v}_2$. The first rotation turns them onto the axes, the stretch lengthens one by 2 and shortens the other by
half, and the last rotation turns them into $2\mathbf{u}_1$ and $0.5\mathbf{u}_2$, the axes of the final ellipse (equation 7).

In [ ]:
C = DSP_COLORS
R = lambda deg: np.array([[np.cos(np.radians(deg)), -np.sin(np.radians(deg))],
                          [np.sin(np.radians(deg)),  np.cos(np.radians(deg))]])
V, S, U = R(30), np.diag([2.0, 0.5]), R(60)
A = U @ S @ V.T
th = np.linspace(0, 2*np.pi, 241); circle = np.vstack([np.cos(th), np.sin(th)])
n_frames = 30                                     # frames 0-9: V^T, 10-19: Sigma, 20-29: U

def M_of(k):
    s = (k % 10 + 1) / 10
    if k < 10:
        return R(-30*s), f'step 1: $V^T$ rotates by {-30*s:.0f}°'
    if k < 20:
        return np.diag([1 + s, 1 - 0.5*s]) @ V.T, f'step 2: $\\Sigma$ stretches by ({1 + s:.2f}, {1 - 0.5*s:.2f})'
    return R(60*s) @ S @ V.T, f'step 3: $U$ rotates by {60*s:.0f}°'

fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot(circle[0], circle[1], '--', color=C['unit_circle'], lw=1, label='unit circle')
fin = A @ circle
ax.plot(fin[0], fin[1], color=C['neutral'], lw=1, alpha=0.35, label=r'final ellipse $\mathbf{A}\cdot$circle')
curve, = ax.plot([], [], color=C['time_domain'], lw=2.2)
arr1 = ax.annotate('', xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color=C['freq_domain'], lw=2.2))
arr2 = ax.annotate('', xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color=C['impulse_resp'], lw=2.2))
ax.plot([], [], color=C['freq_domain'], lw=2, label=r'image of $\mathbf{v}_1$')
ax.plot([], [], color=C['impulse_resp'], lw=2, label=r'image of $\mathbf{v}_2$')
ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
ax.set_aspect('equal'); ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
ax.legend(loc='lower right', fontsize=8)
title = ax.set_title('', fontsize=11)

def update(k):
    M, label = M_of(k)
    pts = M @ circle
    curve.set_data(pts[0], pts[1])
    p1, p2 = M @ V[:, 0], M @ V[:, 1]
    arr1.xy = (p1[0], p1[1]); arr2.xy = (p2[0], p2[1])
    title.set_text(label)
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=250, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 8.4.</b> The SVD as rotate–stretch–rotate, for $\mathbf{A} = \begin{pmatrix}1.0825 & 0.1250\\ 1.3750 & 1.0825\end{pmatrix}$, whose singular values are 2 and 0.5. The blue curve is the image of the unit circle (dashed) after each step; the red and green arrows are the images of $\mathbf{v}_1$ and $\mathbf{v}_2$. Because $\sigma_1\sigma_2 = 1 = \det\mathbf{A}$, the ellipse has the same area as the circle. 30 frames, ten per step. <i>(Python-generated.)</i>

<a name="8.4.2"></a>
### 8.4.2 Singular values are gains

Equation (7) says that an input along $\mathbf{v}_i$ comes out along $\mathbf{u}_i$, multiplied by the <i>gain</i> $\sigma_i$. Any input is a
combination of the $\mathbf{v}_i$, so a matrix amplifies some parts of a signal and suppresses others, and $\sigma_i = 0$ means that part is lost entirely.
When $\mathbf{A}$ is symmetric ($\mathbf{A}^T = \mathbf{A}$), the input and output directions coincide up to sign: $\mathbf{A}\mathbf{v}_i = \lambda_i\mathbf{v}_i$
with $\sigma_i = |\lambda_i|$. A direction that a matrix only stretches is an <b>eigenvector</b>, and the stretch $\lambda_i$ is its eigenvalue.

<div class="alert alert-info">
<b>Proof sketch:</b> <i>Why $\mathbf{A}^T\mathbf{A}$ gives the singular values.</i> Apply $\mathbf{A}^T$ to the first relation in equation (7):
$\mathbf{A}^T\mathbf{A}\mathbf{v}_i = \sigma_i\mathbf{A}^T\mathbf{u}_i = \sigma_i^2\mathbf{v}_i$. So the $\mathbf{v}_i$ are eigenvectors of the symmetric
matrix $\mathbf{A}^T\mathbf{A}$ with eigenvalues $\sigma_i^2$. These eigenvalues cannot be negative, because
$\sigma_i^2 = \mathbf{v}_i^T\mathbf{A}^T\mathbf{A}\mathbf{v}_i = \|\mathbf{A}\mathbf{v}_i\|^2 \ge 0$; that is why singular values are taken $\ge 0$.
Numerical libraries (<code>np.linalg.svd</code>) use more stable algorithms, but this is what they compute.
</div>

<a name="8.4.3"></a>
### 8.4.3 The SVD of a convolution matrix

For convolution the gains have a familiar meaning. If the convolution is <i>periodic</i> (circular, as the DFT assumes), the singular vectors are sampled
complex exponentials $e^{i2\pi kn/N}$, and the singular values are exactly the magnitudes of the DFT of $h$:

$$\sigma_k = |H_k|, \qquad H_k = \sum_n h[n]\,e^{-i2\pi kn/N} \tag{8}$$

For ordinary (linear) convolution the end effects spoil the exact equality, but the singular values still approach samples of the frequency response as $N$
grows,

$$\sigma_k \approx |H(\omega_k)| \tag{9}$$

For the two-tap pulse $h = [1, -0.5]$ the result is exact on a shifted grid of frequencies:

$$\sigma_k^2 = 1.25 - \cos\frac{k\pi}{N+1} = \left|H\!\left(\tfrac{k\pi}{N+1}\right)\right|^2, \qquad k = 1, \dots, N \tag{10}$$

In [ ]:
C = DSP_COLORS
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
# (a) circulant (periodic) convolution, N = 64
N = 64; gp = np.zeros(N); gp[:50] = g
Cm = np.stack([np.roll(gp, j) for j in range(N)], axis=1)
sc = np.sort(np.linalg.svd(Cm, compute_uv=False))[::-1]
Hd = np.sort(np.abs(np.fft.fft(gp)))[::-1]
axs[0].semilogy(Hd, '-', color=C['freq_domain'], label='sorted $|$DFT$(g)|$')
axs[0].semilogy(sc, 'o', color=C['time_domain'], ms=4, mfc='none', label=r'sorted $\sigma_k$')
axs[0].set_xlabel('rank $k$'); axs[0].set_title('(a) Periodic convolution: exact'); axs[0].legend(fontsize=9)
# (b) two-tap linear convolution, N = 16: sigma_k = |H| on the shifted grid k pi/(N+1)
h2, N = np.array([1, -0.5]), 16
s2 = np.sort(np.linalg.svd(convmat(h2, N), compute_uv=False))
w = np.linspace(0, np.pi, 400); wk = np.arange(1, N + 1)*np.pi/(N + 1); wd = 2*np.pi*np.arange(N//2 + 1)/N
Hw = lambda w: np.abs(1 - 0.5*np.exp(-1j*w))
axs[1].plot(w, Hw(w), '-', color=C['freq_domain'], label=r'$|H(\omega)|$')
axs[1].plot(wk, s2, 'o', color=C['time_domain'], ms=6, mfc='none', label=r'$\sigma_k$ at $\omega_k = k\pi/(N+1)$')
axs[1].plot(wd, Hw(wd), 'x', color=C['highlight'], ms=7, mew=2, label='$|H|$ on the DFT grid')
axs[1].set_xlabel(r'$\omega$ [rad/sample]'); axs[1].set_title('(b) Two-tap, $N=16$: shifted grid'); axs[1].legend(fontsize=9, loc='lower right')
axs[1].set_xlim(0, np.pi); axs[1].set_ylim(0.45, 1.55)
# (c) the 50-tap Gaussian of Module 7, linear convolution, N = 300
wk = np.arange(1, nbox + 1)*np.pi/(nbox + 1)
Gw = np.abs(np.exp(-1j*np.outer(wk, np.arange(50))) @ g)
axs[2].semilogy(np.sort(Gw)[::-1], '-', color=C['freq_domain'], label=r'sorted $|G(\omega_k)|$')
axs[2].semilogy(np.arange(0, nbox, 6), sg[::6], 'o', color=C['time_domain'], ms=4, mfc='none', label=r'sorted $\sigma_k$ (every 6th)')
axs[2].set_xlabel('rank $k$'); axs[2].set_title('(c) Gaussian blur, $N=300$'); axs[2].legend(fontsize=9)
axs[2].set_ylim(1e-18, 3)
plt.tight_layout(); plt.show()

<b>Figure 8.5.</b> Singular values are frequency-response magnitudes. (a) Periodic convolution with the 50-tap Gaussian, $N = 64$: the sorted singular values equal the sorted $|$DFT$(g)|$ (largest difference 9e-16). (b) Linear convolution with $h = [1, -0.5]$, $N = 16$: the singular values (circles) lie exactly on $|H(\omega)|$ at $\omega_k = k\pi/(N+1)$, close to but not on the DFT grid (crosses); the largest gap to the DFT-grid values is 0.068 at $N = 16$ and shrinks like $1/N$ (0.0011 at $N = 1024$). (c) Linear convolution with the Gaussian of Figure 8.1, $N = 300$: the singular values track $|G(\omega_k)|$ from 0.9996 down to $10^{-17}$ (median difference 2e-05). <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions.</b> For periodic convolution the singular vectors of $\mathbf{H}$ are the sampled complex
exponentials of Module 3, and the singular values are $|H(\omega)|$. This is the matrix statement of Module 7's eigenfunction property of LTI systems:
a complex exponential goes in and comes out multiplied by a number. For ordinary convolution the end effects spoil this slightly, but only at the edges, and
the agreement improves as $N$ grows (Figure 8.5 b).
</div>

<a name="8.5"></a>
## 8.5 Rank, Conditioning and the Pseudo-inverse

Write the data in the output directions and the solution in the input directions. Because $\mathbf{H}\mathbf{v}_i = \sigma_i\mathbf{u}_i$, the
least-squares solution of $\mathbf{y} = \mathbf{Hx}$ is

$$\mathbf{x} = \sum_i \frac{\mathbf{u}_i^T\mathbf{y}}{\sigma_i}\,\mathbf{v}_i \tag{11}$$

With noisy data $\mathbf{y} = \mathbf{H}\mathbf{x}_{true} + \mathbf{n}$, the solution picks up the extra term

$$\sum_i \frac{\mathbf{u}_i^T\mathbf{n}}{\sigma_i}\,\mathbf{v}_i \tag{12}$$

so every noise component is divided by its singular value. The ratio of the largest to the smallest gain, the <b>condition number</b>

$$\kappa = \sigma_1/\sigma_N \tag{13}$$

bounds how much the relative noise can be amplified. For the Gaussian blur of Figure 8.1, $\kappa = 1.9e+16$: of its 300 singular values, 73 exceed $10^{-1}$,
104 exceed $10^{-2}$, 128 exceed $10^{-3}$, and only 182 exceed $10^{-6}$. Numerically the matrix is singular.

The <b>truncated SVD</b> (TSVD) keeps only the $k$ largest singular values,

$$\mathbf{x}_k = \sum_{i=1}^{k} \frac{\mathbf{u}_i^T\mathbf{y}}{\sigma_i}\,\mathbf{v}_i \tag{14}$$

trading a little resolution (the discarded directions) for stability. The Fourier-domain versions of the same idea replace $|G(\omega)|$ by a floor
$\epsilon$ wherever it is smaller (<b>water level</b>), or divide by $G$ with a damping term:

$$\widehat{X}(\omega) = \frac{\widehat{Y}(\omega)}{\max\big(|G(\omega)|, \epsilon\big)\,e^{i\arg G(\omega)}}, \qquad
\widehat{X}(\omega) = \frac{\overline{G(\omega)}\,\widehat{Y}(\omega)}{|G(\omega)|^2 + \epsilon^2} \tag{15}$$

<div class="alert alert-info">
<b>Note:</b> <i>Numerical rank.</i> In exact arithmetic the rank of a matrix is the number of nonzero singular values. In floating point, tiny singular
values are indistinguishable from rounding error, so <code>np.linalg.matrix_rank</code> counts only those above a tolerance (about
$\sigma_1 \times N \times 2.2\times 10^{-16}$). For the Gaussian blur that tolerance is crossed long before index 300.
</div>

In [ ]:
C = DSP_COLORS
c = (Ug.T @ y_obs) / sg                                  # coefficients u_i^T y / sigma_i
Xk = np.cumsum(Vgt.T * c, axis=1)                        # column k-1 = TSVD solution with k terms
errs = np.linalg.norm(Xk - x_box[:, None], axis=0) / np.linalg.norm(x_box)
kopt = int(np.argmin(errs) + 1)
Nf = len(y_obs); G = np.fft.fft(g, Nf); Y = np.fft.fft(y_obs)
wl = 0.03*np.abs(G).max(); Gd = np.where(np.abs(G) < wl, wl*np.exp(1j*np.angle(G)), G)
x_wl = np.real(np.fft.ifft(Y / Gd))[:nbox]
eps = 0.01*np.abs(G).max(); x_dm = np.real(np.fft.ifft(Y*np.conj(G) / (np.abs(G)**2 + eps**2)))[:nbox]
rel = lambda v: np.linalg.norm(v - x_box) / np.linalg.norm(x_box)
fig, axs = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True)
axs[0, 0].plot(x_box, color=C['input'], label='truth'); axs[0, 0].plot(y_obs[:nbox], color=C['output'], label='data')
axs[0, 0].set_title('(a) Truth and data'); axs[0, 0].legend(loc='upper right', fontsize=9)
axs[0, 1].plot(Xk[:, -1], color=C['output'], lw=0.8); axs[0, 1].set_ylim(-3, 3)
axs[0, 1].set_title(f'(b) All {nbox} singular values: error {errs[-1]:.0e} (clipped)')
axs[1, 0].plot(x_box, color=C['neutral'], lw=1); axs[1, 0].plot(Xk[:, kopt - 1], color=C['time_domain'])
axs[1, 0].set_title(f'(c) Truncated SVD, k = {kopt}: error {errs[kopt - 1]:.3f}')
axs[1, 1].plot(x_box, color=C['neutral'], lw=1)
axs[1, 1].plot(x_wl, color=C['freq_domain'], label=f'water level: {rel(x_wl):.3f}')
axs[1, 1].plot(x_dm, color=C['highlight'], ls='--', label=f'damped: {rel(x_dm):.3f}')
axs[1, 1].set_title('(d) Spectral division, regularized'); axs[1, 1].legend(loc='upper right', fontsize=9)
for ax in axs.flat:
    ax.set_xlim(0, nbox - 1)
for ax in axs[0]:
    ax.tick_params(labelbottom=True)
for ax in axs[1]:
    ax.set_xlabel('sample $n$'); ax.set_ylim(-0.4, 1.45)
axs[0, 0].set_ylim(-0.4, 1.45)
plt.tight_layout(); plt.show()

<b>Figure 8.6.</b> Stabilized deconvolution of the data in Figure 8.1. (a) Truth and data. (b) The full inverse, all 300 singular values: relative error 4e+12 (clipped to $\pm 3$). (c) Truncated SVD with the best $k = 97$ (here $\sigma_k = 0.019$): relative error 0.099. (d) Water-level division with $\epsilon = 0.03\max|G|$ (error 0.112) and damped division with $\epsilon = 0.01\max|G|$ (error 0.100). The truth is drawn in gray in (c) and (d). <i>(Python-generated.)</i>

How was $k$ chosen? Figure 8.7 plots the error of $\mathbf{x}_k$ against $k$. Adding terms first sharpens the boxcar; once $\sigma_k$ falls toward
the noise level, each new term adds more amplified noise than signal and the error climbs steeply. With real data the truth is unknown, so $k$ is chosen from
the data alone, for example by stopping when the residual $\|\mathbf{y} - \mathbf{Hx}_k\|$ reaches the expected noise size.

In [ ]:
C = DSP_COLORS
c = (Ug.T @ y_obs) / sg
errs = np.linalg.norm(np.cumsum(Vgt.T * c, axis=1) - x_box[:, None], axis=0) / np.linalg.norm(x_box)
kopt = int(np.argmin(errs) + 1); k = np.arange(1, nbox + 1)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.semilogy(k, errs, color=C['time_domain'], label='relative error of the TSVD solution')
ax.axvline(kopt, color=C['highlight'], ls='--', lw=1.5)
ax.text(kopt + 4, 30, f'$k_{{opt}}$ = {kopt}', color=C['highlight'], fontsize=11)
ax.set_xlabel('number of singular values kept, $k$'); ax.set_ylabel('relative error', color=C['time_domain'])
ax.set_ylim(3e-2, 1e14); ax.set_yticks(10.0**np.arange(-1, 14, 3)); ax.set_xlim(0, nbox)
ax2 = ax.twinx()
ax2.semilogy(k, sg, color=C['freq_domain'], lw=1.5, label=r'$\sigma_k$')
ax2.axhline(1e-3, color=C['noise'], ls=':', lw=1.5)
ax2.text(8, 1.6e-3, 'noise rms $10^{-3}$', color=C['noise'], fontsize=10)
ax2.set_ylabel(r'singular value $\sigma_k$', color=C['freq_domain']); ax2.set_ylim(1e-18, 3); ax2.set_yticks(10.0**np.arange(-15, 1, 3)); ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

<b>Figure 8.7.</b> Choosing the truncation. Relative error of the TSVD solution (blue, left axis) and the singular values (red, right axis) against the number of singular values kept. The error falls to its minimum, 0.099 at $k = 97$, then rises to 3.9 at $k = 150$ and 4e+03 at $k = 200$, as $\sigma_k$ drops below the noise level (olive line). <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication.</b> Truncated SVD and water-level spectral division are the same idea in two bases. TSVD discards the
directions whose gain $\sigma_i$ is below a threshold; water level refuses to divide by $|G(\omega)|$ below a threshold. Because $\sigma_i \approx |G(\omega_i)|$
(Figure 8.5), their errors in Figure 8.6 are nearly the same (0.099 and 0.112). Module 7's constant <code>DC_term</code> added to $g$
was a crude water level.
</div>

<a name="8.6"></a>
## 8.6 Low-Rank Approximation

Equation (6) can be written as a sum of rank-1 pieces, $\mathbf{A} = \sum_i \sigma_i\,\mathbf{u}_i\mathbf{v}_i^T$, each an outer product of an output
direction and an input direction. Keeping only the first $k$,

$$\mathbf{A}_k = \sum_{i=1}^{k} \sigma_i\,\mathbf{u}_i\mathbf{v}_i^T \tag{16}$$

gives the best possible rank-$k$ approximation (the Eckart–Young theorem; Eckart and Young, 1936), with error

$$\|\mathbf{A} - \mathbf{A}_k\|_F^2 = \sum_{i>k}\sigma_i^2 \tag{17}$$

where $\|\mathbf{M}\|_F^2 = \sum_{mn} M_{mn}^2$ is the total energy of the matrix. The fraction of the energy that $\mathbf{A}_k$ captures is therefore

$$\frac{\sum_{i\le k}\sigma_i^2}{\sum_i \sigma_i^2} \tag{18}$$

An $m\times n$ image stores $mn$ numbers; $\mathbf{A}_k$ stores $k$ vectors $\mathbf{u}_i$, $k$ vectors $\mathbf{v}_i$ and $k$ numbers $\sigma_i$:

$$k\,(m + n + 1) \text{ numbers} \tag{19}$$

which saves space only while $k < mn/(m + n + 1)$.

<div class="alert alert-info">
<b>Proof sketch:</b> <i>The Eckart–Young theorem (sketch).</i> The rank-1 pieces $\mathbf{u}_i\mathbf{v}_i^T$ are orthogonal to one another when the matrix
entries are treated as one long vector, because the $\mathbf{u}_i$ are orthogonal and so are the $\mathbf{v}_i$. Energies of orthogonal pieces add, exactly as
the energies of Fourier components add in Parseval's relation (Module 5), so the energy of $\mathbf{A} - \mathbf{A}_k$ is $\sum_{i>k}\sigma_i^2$. Showing
that <i>no other</i> rank-$k$ matrix does better takes a little more work; the key step is that any rank-$k$ matrix must miss some combination of the first
$k+1$ input directions, and that combination costs at least $\sigma_{k+1}^2$.<br><br><i>Reference:</i> C. Eckart and G. Young (1936), The approximation
of one matrix by another of lower rank, <i>Psychometrika</i>, 1(3), 211&ndash;218.
</div>

Figure 8.8 reconstructs a $260\times 260$ photograph of a brick wall (Module 6) from its first $k$ singular values, with $k$ running from 1 to
130. Watch how early the image is recognizable: the first few terms capture the regular rows of bricks, because a repeating pattern is close to a sum of a
few outer products. The title shows the storage of equation (19) and the error of equation (17). At $k = 130$ the storage reaches 100%, the
break-even point, so compression has stopped paying.

In [ ]:
C = DSP_COLORS
B = gray('Fig/2.3_BRICKS.jpg', 2)                        # 260 x 260 grayscale, values 0-1
Ub, sb, Vbt = np.linalg.svd(B, full_matrices=False)
m, n = B.shape
ks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 15, 17, 19, 22, 25, 28, 32, 36, 41, 47, 53, 60, 69, 78, 89, 101, 114, 130]   # 30 frames, ~log-spaced
n_frames = len(ks)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(7.8, 3.3), gridspec_kw=dict(width_ratios=[1, 1.25]))
im = axa.imshow(np.zeros_like(B), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axa.set_xticks([]); axa.set_yticks([]); axa.grid(False)
axb.semilogy(np.arange(1, len(sb) + 1), sb, color=C['neutral'], lw=1.2, label=r'all $\sigma_k$')
kept, = axb.semilogy([], [], color=C['freq_domain'], lw=3, marker='o', ms=6, markevery=[0], label='kept')
axb.set_xlabel('index $k$'); axb.set_ylabel(r'$\sigma_k$'); axb.legend(loc='upper right', fontsize=9)
axb.set_xlim(0, len(sb))
title = fig.suptitle('', fontsize=11)

def update(i):
    k = ks[i]
    im.set_data(np.clip((Ub[:, :k]*sb[:k]) @ Vbt[:k], 0, 1))
    kept.set_data(np.arange(1, k + 1), sb[:k])
    err = np.sqrt((sb[k:]**2).sum() / (sb**2).sum())
    title.set_text(f'rank k = {k}:  storage {100*k*(m + n + 1)/(m*n):.0f}% of the image,  relative error {100*err:.1f}%')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=450, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 8.8.</b> Rank-$k$ approximation of the bricks photograph ($260\times 260$ pixels, grayscale). (a) The image $\mathbf{A}_k$. (b) The singular values, with the kept ones in red. The relative error in the title is $\sqrt{\sum_{i>k}\sigma_i^2/\sum_i\sigma_i^2}$ (equation 17). 30 frames, $k$ from 1 to 130 in roughly logarithmic steps. <i>(Python-generated.)</i>

Not every image is as compressible. Figure 8.9 compares the bricks with the New York photograph of Module 6: with the mean removed, 90% of the
bricks' energy is in its first 39 singular values, while the city at night needs 108.

In [ ]:
C = DSP_COLORS
imgs = {'bricks': gray('Fig/2.3_BRICKS.jpg', 2), 'New York at night': gray('Fig/2.3_NEW_YORK.png', 2)}
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.4))
for (name, img), col in zip(imgs.items(), [C['time_domain'], C['freq_domain']]):
    s = np.linalg.svd(img - img.mean(), compute_uv=False)
    cum = np.cumsum(s**2) / np.sum(s**2); k90 = int(np.searchsorted(cum, 0.9) + 1)
    axa.semilogy(np.arange(1, len(s) + 1), s/s[0], color=col, label=f'{name} ({img.shape[0]}×{img.shape[1]})')
    axb.plot(np.arange(1, len(s) + 1), cum, color=col, label=f'{name}: 90% at k = {k90}')
axb.axhline(0.9, color=C['neutral'], ls=':', lw=1.2)
axa.set_xlabel('index $k$'); axa.set_ylabel(r'$\sigma_k/\sigma_1$'); axa.set_title('(a) Singular values (mean removed)'); axa.legend(fontsize=9)
axb.set_xlabel('rank $k$'); axb.set_ylabel('fraction of energy'); axb.set_title('(b) Cumulative energy'); axb.legend(loc='lower right', fontsize=9)
axb.set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

<b>Figure 8.9.</b> Why some images compress better. (a) Singular values, normalized by the largest, for the bricks (260×260) and New York (341×512) photographs, mean removed. (b) Cumulative energy fraction (equation 18): 90% at $k = 39$ for the bricks and $k = 108$ for New York. The regular brick pattern is far more compressible than the scattered lights of the city. <i>(Python-generated.)</i>

<a name="8.7"></a>
## 8.7 Geophysical Application: SVD Filtering of a Shot Gather

<a name="8.7.1"></a>
### 8.7.1 A synthetic gather

A seismic shot gather is a matrix: rows are time samples and columns are traces. A reflection that arrives at the same time on every trace is the same
column repeated, so it is <b>rank 1</b>, while random noise spreads its energy over all the singular values. Figure 8.10 tests this on a synthetic gather.

In [ ]:
C = DSP_COLORS
nt, ntr, dt = 250, 60, 0.004
t = np.arange(nt)*dt
flat = np.zeros((nt, ntr))
for t0, amp in ((0.3, 1.0), (0.5, -0.7), (0.7, 0.5)):          # three flat 20 Hz reflections
    flat += amp*ricker(t - t0, 20)[:, None]
dip = np.stack([0.8*ricker(t - (0.15 + 0.008*j), 20) for j in range(ntr)], axis=1)   # 8 ms per trace
noise = 0.3*np.random.default_rng(404).standard_normal((nt, ntr))
data = flat + dip + noise
U, s, Vt = np.linalg.svd(data, full_matrices=False)
r1 = s[0]*np.outer(U[:, 0], Vt[0])
fig = plt.figure(figsize=(12.5, 3.9))
gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 1.3])
ext = [0, ntr, t[-1], 0]
for i, (img, ttl) in enumerate([(flat + dip, '(a) clean'), (data, '(b) noisy'), (r1, '(c) rank 1'), (data - r1, '(d) (b) minus (c)')]):
    ax = fig.add_subplot(gs[i])
    ax.imshow(img, cmap='gray', vmin=-1, vmax=1, aspect='auto', extent=ext)
    ax.set_title(ttl, fontsize=12); ax.set_xlabel('trace'); ax.grid(False)
    if i == 0:
        ax.set_ylabel('time [s]')
    else:
        ax.set_yticklabels([])
ax = fig.add_subplot(gs[4])
for M, lab, col in [(data, 'noisy data', C['output']), (noise, 'noise alone', C['noise']), (flat, 'flat events', C['time_domain']),
                    (dip, 'dipping event', C['highlight'])]:
    sv = np.linalg.svd(M, compute_uv=False)
    ax.semilogy(np.arange(1, ntr + 1), np.maximum(sv, 1e-16), 'o-', ms=3, lw=1, color=col, label=lab)
ax.set_ylim(1e-2, 2e2); ax.set_xlabel('index $k$'); ax.set_title(r'(e) $\sigma_k$', fontsize=12); ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

<b>Figure 8.10.</b> SVD filtering of a synthetic gather (250 samples × 60 traces, $\Delta t = 4$ ms). (a) Three flat 20 Hz reflections and one dipping event (8 ms per trace). (b) With random noise added: signal-to-noise ratio -4.0 dB. (c) The rank-1 approximation of (b) recovers the flat reflections with a signal-to-noise ratio of 10.3 dB, but keeps only 0.05% of the dipping event's energy. (d) What rank 1 removed: the noise and the dipping event. (e) Singular values: the flat events are exactly rank 1, while the dipping event needs 23 singular values for 90% of its energy and the noise spreads over all 60. <i>(Python-generated.)</i>

<a name="8.7.2"></a>
### 8.7.2 A real gather: CAPshot

Real data are less obliging. The CAPshot record of Module 5 is a land shot recorded on a split spread, with the source near trace 150. Its strongest
arrivals are the first breaks, whose arrival time grows linearly with distance from the source, so they dip across the gather and are not rank 1. A
<b>linear-moveout</b> (LMO) correction shifts each trace earlier by an amount proportional to its distance from the source,

$$t' = t - p\,|x - x_s| \tag{20}$$

with $p$ in seconds per trace. We choose $p$ by scanning for the value that puts the largest fraction of the energy into the first singular value, then
apply rank-$k$ filtering to a 0.3 s window under the flattened first breaks, after adding noise of known size so that the result can be measured.

In [ ]:
C = DSP_COLORS
import read_in_rsf as sf                                     # RSF reader in the repository root
a = sf.read_rsf_file('data/CAPshot', use_memmap=True)
d = np.array(a.data).astype(float); dt = a.view.dx[0]       # 2500 samples x 300 traces, dt = 4 ms
tr = np.arange(10, 145)                                      # left half of the split spread (source near trace 150)
w = d[:500, 10:145].copy()                                   # 0-2 s
w /= np.sqrt((w**2).mean(axis=0)) + 1e-30                    # balance the traces
cl = np.percentile(np.abs(w), 99.5); w = np.clip(w, -cl, cl)

def lmo(W, p, xs=150):
    """Linear-moveout correction: shift trace x up by p*|xs - x| seconds."""
    out = np.zeros_like(W); nt = W.shape[0]
    for j, x in enumerate(tr):
        s = int(round(p*(xs - x)/dt))
        if s < nt:
            out[:nt - s, j] = W[s:, j]
    return out

def frac1(W):
    s = np.linalg.svd(W, compute_uv=False)
    return s[0]**2 / (s**2).sum()

ps = np.round(np.arange(0.0030, 0.01001, 0.000125), 6)
fr = np.array([frac1(lmo(w, p)) for p in ps]); pbest = ps[np.argmax(fr)]
f = lmo(w, pbest)
win = f[24:99]                                               # 0.3 s window under the flattened first arrivals
noise = np.random.default_rng(404).standard_normal(win.shape)
noise *= np.linalg.norm(win) / np.linalg.norm(noise) / 10**(6/20)    # SNR 6 dB
noisy = win + noise
U, s, Vt = np.linalg.svd(noisy, full_matrices=False)
snr = lambda est: 20*np.log10(np.linalg.norm(win) / np.linalg.norm(est - win))
best = max(range(1, 41), key=lambda k: snr((U[:, :k]*s[:k]) @ Vt[:k]))
rk = (U[:, :best]*s[:best]) @ Vt[:best]
fig = plt.figure(figsize=(12, 6.6))
gs = fig.add_gridspec(2, 3, height_ratios=[1.15, 1])
ax = fig.add_subplot(gs[0, :2])
ax.imshow(w.T, cmap='gray', vmin=-2, vmax=2, aspect='auto', extent=[0, 2, tr[-1], tr[0]])
ax.set_xlabel('time [s]'); ax.set_ylabel('trace'); ax.set_title('(a) CAPshot, left spread, balanced (time across)'); ax.grid(False)
ax = fig.add_subplot(gs[0, 2])
ax.plot(ps*1000, fr, 'o-', color=C['time_domain'], ms=3)
ax.axvline(pbest*1000, color=C['highlight'], ls='--')
ax.text(9.95, fr.min(), f'best p = {pbest*1000:.2f} ms/trace', color=C['highlight'], fontsize=10, ha='right', va='bottom')
ax.set_xlabel('LMO slope $p$ [ms/trace]'); ax.set_ylabel(r'$\sigma_1^2/\sum\sigma_k^2$'); ax.set_title('(b) Flatness scan')
tw = np.arange(24, 99)*dt
for i, (img, ttl) in enumerate([(noisy, f'(c) flattened + noise: {snr(noisy):.1f} dB'),
                                (rk, f'(d) rank {best}: {snr(rk):.1f} dB'), (noisy - rk, '(e) (c) minus (d)')]):
    ax = fig.add_subplot(gs[1, i])
    ax.imshow(img, cmap='gray', vmin=-3, vmax=3, aspect='auto', extent=[tr[0], tr[-1], tw[-1], tw[0]])
    ax.set_title(ttl, fontsize=12); ax.set_xlabel('trace'); ax.grid(False)
    if i == 0:
        ax.set_ylabel('flattened time [s]')
plt.tight_layout(); plt.show()

<b>Figure 8.11.</b> SVD filtering of the CAPshot gather. (a) The left half of the spread (traces 10–144), 0–2 s, with each trace balanced to unit rms; time runs across. (b) The fraction of energy in the first singular value against the LMO slope $p$: it rises from 0.059 (no correction) to 0.141 at $p = 6.25$ ms/trace. (c) A 0.3 s window after LMO, with noise added for a signal-to-noise ratio of 6.0 dB. (d) Its best rank-$k$ approximation, $k = 20$: 8.4 dB. (e) What was removed. The gain is real but modest: even after flattening, the first breaks are not one repeated waveform. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example.</b> The first breaks in CAPshot arrive later with distance from the source along a straight line,
the refracted head waves of POTE1 (GPGN 328): their slope $p$ is the inverse of an apparent velocity. Flattening them with a time shift $t - p|x - x_s|$ (a
convolution with a shifted impulse, Module 7) is what turns a dipping event into a nearly rank-1 one. Converting $p$ from seconds per trace into a velocity
needs the receiver spacing, which the file does not record.
</div>

<div class="alert alert-success">
<h3><center>Distraction A: Dipping and curved events</center></h3>

Linear moveout flattens events whose arrival time grows linearly with offset, such as head waves and
ground roll. Reflections from flat layers arrive on a hyperbola, $t(x) = \sqrt{t_0^2 + x^2/v^2}$, and are flattened instead by a normal-moveout (NMO)
correction, which needs the velocity $v$ (PHGN 100 kinematics with the Pythagorean theorem). Processing systems scan velocities to find it, exactly as we
scanned $p$ in Figure 8.11 b. Dipping or curved events that cannot be flattened by one correction stay spread over many singular values, which limits SVD
filtering of real gathers.
</div>

<a name="8.8"></a>
## 8.8 Parting Thoughts

Once signals are vectors, every linear system is a matrix, and the SVD reads any matrix as rotate, stretch and rotate. The singular values are the gains:
for convolution they are (approximately) the frequency response, so small gains mean frequencies the system destroys, and inverting them amplifies noise.
Truncating the SVD, or putting a floor under $|G(\omega)|$, keeps the stable part of the inverse, and keeping only the large singular values compresses
images and separates coherent signal from noise whenever the signal is close to rank-$k$. In Module 11 the DFT matrix appears as the operator whose columns
are the circulant singular vectors of equation (8), the sampled complex exponentials.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 8.1 &mdash; A 3×3 convolution matrix and its adjoint</b>

A causal two-tap system has $h[0] = 2$, $h[1] = 1$ and acts on 3-sample signals, keeping the first three outputs: $y[n] = 2x[n] + x[n-1]$, with $x[-1] = 0$. Take $\mathbf{x} = (3, 1, -2)$ and a second vector $\mathbf{y} = (1, 1, 1)$.

<b>(a)</b> Write the $3\times 3$ matrix $\mathbf{H}$ (Section 8.2) and compute $\mathbf{Hx}$.

<b>(b)</b> Compute $\mathbf{H}^T\mathbf{y}$ and show that it equals the crosscorrelation $(\mathbf{H}^T\mathbf{y})[n] = 2y[n] + y[n+1]$ (with $y[3] = 0$) of Module 7. Verify the dot-product test $\langle\mathbf{Hx}, \mathbf{y}\rangle = \langle\mathbf{x}, \mathbf{H}^T\mathbf{y}\rangle$, using the PHGN 100 dot product.

<b>(c)</b> A student claims: "$\mathbf{H}^T$ undoes the shift-and-scale of $\mathbf{H}$, so $\mathbf{H}^T$ is the inverse of $\mathbf{H}$." Test this with $\mathbf{H}^T\mathbf{H}$.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 8.1</b></summary>

<b>(a)</b> $\mathbf{H} = \begin{pmatrix}2&0&0\\1&2&0\\0&1&2\end{pmatrix}$ (lower triangular and constant along its diagonals, a Toeplitz matrix). $\mathbf{Hx} = (6, 5, -3)$.

<b>(b)</b> $\mathbf{H}^T = \begin{pmatrix}2&1&0\\0&2&1\\0&0&2\end{pmatrix}$, so $\mathbf{H}^T\mathbf{y} = (3, 3, 2)$, and $2y[n] + y[n+1]$ gives $(2+1, 2+1, 2+0) = (3, 3, 2)$, the same. $\langle\mathbf{Hx}, \mathbf{y}\rangle = 6 + 5 - 3 = 8$ and $\langle\mathbf{x}, \mathbf{H}^T\mathbf{y}\rangle = 9 + 3 - 4 = 8$.

<b>(c)</b> $\mathbf{H}^T\mathbf{H} = \begin{pmatrix}5&2&0\\2&5&2\\0&2&4\end{pmatrix} \neq \mathbf{I}$. The adjoint is not the inverse: it correlates (a matched filter, Figure 8.3) and does not deconvolve. Here the singular values are 2.762, 2.125 and 1.363 ($\kappa = 2.03$), so the true inverse is well behaved, but it is not $\mathbf{H}^T$.
</details>

<div class="alert alert-warning">
<b>Microquestion 8.2 &mdash; How ill-conditioned is an RC instrument?</b>

A recorder is modeled as a sampled PHGN 200 RC low-pass filter, $y[n] = a\,y[n-1] + (1-a)\,x[n]$ with $a = 0.9$, so $h[n] = (1-a)a^n$ for $n \ge 0$ and $|H(\omega)| = (1-a)/|1 - a e^{-i\omega}|$. Its long convolution matrix is to be inverted from data with noise rms $10^{-3}$. For comparison, the Gaussian blur of Figure 8.6 has $\kappa \approx 10^{16}$.

<b>(a)</b> Using $\sigma_k \approx |H(\omega_k)|$ (equation 9), estimate $\sigma_{max}$, $\sigma_{min}$ and $\kappa$ for the RC matrix.

<b>(b)</b> Estimate the worst-case noise amplification for each system, and decide which one needs truncation. Show that the RC inverse is itself a short filter, $x[n] = (y[n] - a\,y[n-1])/(1-a)$, by substituting it into the recursion.

<b>(c)</b> "The RC instrument has $\kappa = 19$, so instrument deconvolution never needs regularization." Give a physical counterexample.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 8.2</b></summary>

<b>(a)</b> $\sigma_{max} \to |H(0)| = 1$ and $\sigma_{min} \to |H(\pi)| = (1-a)/(1+a) = 0.1/1.9 = 0.0526$, so $\kappa \to (1+a)/(1-a) = 19$. A matrix with $N = 400$ columns gives 18.95.

<b>(b)</b> Noise times $\kappa$: the RC system gives $10^{-3}\times 19 = 0.019$ (harmless); the Gaussian gives $10^{-3}\times 10^{16}$ (hopeless), so only the Gaussian needs truncation. Substituting, $y[n] - a\,y[n-1] = (1-a)\,x[n]$, which recovers $x$ exactly. This two-tap inverse amplifies white noise by $\sqrt{1 + a^2}/(1-a) = 13.5$, below the bound $\kappa = 19$.

<b>(c)</b> Any instrument whose $|H(\omega)| \approx 0$ inside the signal band: a geophone well below its natural frequency, an anti-alias filter beyond its cutoff, or the band-limited Gaussian of Figure 8.6. $\kappa$ depends on the <i>ratio</i> of the gains, and noise matters only relative to $\sigma_{min}$. The GPR deconvolution question in Module 7 has this answer.
</details>

<div class="alert alert-warning">
<b>Microquestion 8.3 &mdash; Compressing and filtering a shot gather</b>

A 4 s window of the CAPshot gather (Module 5) is a $1000\times 300$ matrix (time samples × traces). With its traces balanced, its first 20 singular values hold 39.6% of the total energy $\sum\sigma_i^2$. The synthetic dipping event of Figure 8.10, crossing 60 traces at 8 ms per trace, needs 23 singular values to capture 90% of its energy, while a flat event is exactly rank 1.

<b>(a)</b> How many numbers does a rank-20 approximation store (equation 19), as a fraction of the full matrix? What relative (Frobenius) error does it have (equation 17)?

<b>(b)</b> If the noise were spread evenly over all 300 singular directions, what fraction of the noise energy would rank 20 keep, and what noise reduction in decibels would that imply? A seeded $1000\times 300$ Gaussian noise matrix actually keeps 14.2% in its top 20. Explain the difference.

<b>(c)</b> Critique: "Keeping the top-$k$ singular values always removes noise and keeps the signal." Use the dipping-event numbers and the PHGN 100 kinematics of moveout ($\Delta t = \Delta x/v_{app}$) in your answer.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 8.3</b></summary>

<b>(a)</b> $k(m + n + 1) = 20\times 1301 = 26\,020$ numbers versus $300\,000$, i.e. 8.7%. The relative error is $\sqrt{1 - 0.396} = 0.777$ (78%), so this gather is <i>not</i> compressible at $k = 20$.

<b>(b)</b> Naively $20/300 = 6.7\%$, a reduction of $10\log_{10}15 = 11.8$ dB. But the SVD sorts directions by size, so the top 20 singular values of a random matrix are its <i>largest</i>: it keeps 14.2%, and the real reduction is $10\log_{10}(1/0.142) = 8.5$ dB. With only 40% of the data's energy in those 20 directions, much of the signal is discarded too.

<b>(c)</b> The claim holds only when the signal is laterally coherent (flat: rank 1). An event with moveout $\Delta t = \Delta x/v_{app}$ shifts from trace to trace, so it is spread over many singular directions (23 for 90% here), and a small $k$ removes it along with the noise: in Figure 8.10, rank 1 keeps only 0.05% of the dipping event. Flatten first (LMO, Figure 8.11), or accept the loss.
</details>